RL 训推 diff 对齐的核心思路是“让训练与推理在相同输入下走同一条数值路径”；
- 排查应遵循“<font color='red'>逐层定位分叉点 → 按模块逐项核对实现与精度</font>”的工程化流程。

## 核心思路
训推不一致（Training-Inference Mismatch）指的是：同一权重、同一 token 序列，在 rollout 推理引擎和训练引擎上**算出的 logprob 不同**，常用 logdiff 衡量。

在 RL 里，rollout 采样来自<font color='red'>推理策略 μ</font>，梯度更新却依赖<font color='red'>训练策略 π</font>；
- <font color='blue'>一旦两者不一致，策略梯度会出现漂移项，**并可能和奖励信号耦合形成正反馈**，最终导致训练崩溃。</font>


对齐通常分三层推进：
| 层级 | 目标 | <font color='red'>典型手段</font> |
|---|---|---|
| **模型层** | 保证权重、结构、并行配置一致 | 统一 TP/DP/EP 切分，避免训练与推理使用不同模型实现 |
| **算子层** | 保证前向数值路径一致 | <font color='red'>统一</font> FA/PFA、RMSNorm、RoPE、softmax、MoE router 等算子 |
| **执行层** | 保证累加顺序、分块、精度、随机性一致 | 锁定 reduce 序、统一分块大小、统一精度转换、消除非确定性 |

更工程化的做法是引入 <font color='red'>逐层精确匹配测试</font>：固定同一输入，<font color='blue'>比较训练与 rollout 每一层 hidden state，从第 0 层开始找到第一个出现 diff 的位置。</font>

## 排查清单
###  先定位分叉点
   - 用同一 prompt + 同一 rollout 序列，分别在训练引擎和推理引擎跑前向。
   - <font color='blue'>逐层比较 hidden state、attention logits、attention weights、norm 输出、logits、logprob。</font>
   - <font color='red'>记录第一次出现 diff 的层、算子和 token 位置。</font>

在 RL（强化学习）的大模型对齐训练中，“先定位分叉点” 是排查训推不一致（Training-Inference Mismatch）最核心、也是最高效的起手式。

**它的核心原理是“控制变量法”：**
  - 既然训练和推理出现了数值差异（Diff），我们就通过固定相同的输入，<font color='red'>让两套引擎从第 0 层开始逐层输出中间结果，从而精准找到它们第一次“分道扬镳”的位置。</font>


下面为你详细拆解这张排查清单的每一个细节及其背后的技术逻辑：
#### 前置条件：固定变量（用同一 prompt + 同一 rollout 序列）
排查的前提是必须确保训练引擎和推理引擎面对的是完全一模一样的计算对象。
  - 同一 Prompt（输入）：
    - 输入给模型的 input_ids、attention_mask 必须严格一致。<font color='red'>如果训练和推理的 prompt 长度、内容或者 token 切分不同，后续的隐藏层状态自然无法比较。</font>
  - 同一 Rollout 序列（固定采样路径）：<font color='red'>这是 RL 场景下最关键的一点。</font>
    - <font color='blue'>在训练时，模型通常会对 rollout 阶段生成的文本重新计算一遍概率（logprob）来计算 PPO 的损失函数。</font>
    - <font color='red'>因此，你必须直接把 Rollout 引擎生成的完整 token 序列 拿过来，作为训练引擎前向传播的输入。这样就能确保两边不仅在 prompt 上一致，在后续的每一个自回归生成步骤、KV Cache 的拼接上都拥有完全相同的 token 序列。</font>
    
#### 核心动作：逐层比较关键中间状态
大模型（Transformer 架构）是<font color='red'>由数十个层（Layer）堆叠而成的</font>。<font color='blue'>如果只看最终的 Loss 或总体的 logprob 差异，很难发现具体是哪个环节出了错。</font>我们必须像“查账”一样，逐层核对它们的中间产出：
  - Hidden State（隐藏层状态）：这是模型的“记忆”和“思维”。对比每一层 Transformer Block 输出的张量。
    - <font color='red'>如果第 i 层的 Hidden State 对齐，而第 i+1 层开始出现微弱的 diff，那么问题就出在第 i+1 层内部的计算上。</font>
  - Attention Logits & Attention Weights（<font color='red'>注意力得分与权重</font>）：
    - Logits 是 <font color='blue'>Softmax 之前</font>的原始分值，
    - Weights 是 <font color='blue'>Softmax 之后</font>的归一化概率。
    - <font color='red'>这是最容易出问题的环节。</font>
      - 训练时通常使用带 mask 的 FlashAttention（一次性处理整个序列），
      - 而推理时使用的是基于 KV Cache 的 PagedAttention 或逐步生成。
      - <font color='blue'>两边的注意力计算路径差异极大。</font>
  - Norm 输出（归一化输出）：对比 LayerNorm 或 RMSNorm 之后的激活值。虽然 Norm 算子本身很简单，但如果两边的实现精度（FP16 vs BF16）或融合算子的实现细节不同，这里也会产生累积误差。
  - <font color='blue'>最终 Logits & Logprob</font>：这是最终对外的数值。
    - 如果前面的 Hidden State 都对齐了，但最终 Logits 或 Logprob 不对齐，说明问题出在最后的 LM Head 计算或 Softmax 精度控制上。
  
#### 目标与结果：记录分叉点（Layer、算子与 Token 位置）

当逐层对比完成后，你会得到三种典型的结论，这也是排查的最终产出：
##### Embedding 层（第 0 层）就出现了 Diff：
  - 诊断：这属于最底层的参数或输入问题。
  - 原因：训练和推理加载的权重文件不一致、Tokenizer 词表对不齐，或者输入到两边的张量数据类型（dtype，如 BF16 vs FP32）不同。
##### 在某一个特定的 Transformer 层（例如 Layer 5）突然出现 Diff：
  - 诊断：问题出在该层特有的结构或算子上。
  - 原因：重点检查该层的 RoPE（旋转位置编码）实现、MoE 模型的路由（Router）机制、特定的融合算子（如 FlashAttention 在两边的分块策略是否一致）。
##### 前面的层 diff 极小，但在某个特定 Token 位置后误差指数级放大：
  - 诊断：这是大模型训练中的“蝴蝶效应”。
  - 原因：由于底层算子（如矩阵乘法的归约顺序 Reduce order）在 BF16 低精度下产生的极微小浮点误差，随着网络层数和序列长度的增加被不断放大。
  
总结来说，“先定位分叉点”就是把一个黑盒子里的复杂计算，拆解成一个个透明的白盒子。找到第一个对不上的白盒子，剩下的修复工作就有了明确的方向。

##### 这是一份完整的、可直接在命令行运行的自动化调试脚本。

它整合了 argparse 命令行解析、自动化的 Hook 注册以及逐层对比逻辑。运行该脚本后，它会自动定位并打印出训练与推理第一次出现数值分叉（Mismatch）的具体层、算子和张量位置，极大地节省了 RL 训推对齐时的排查时间。
你可以将以下代码保存为 debug_layerwise_diff.py

完整的自动化排查脚本 (debug_layerwise_diff.py)

In [ ]:
import argparse
import torch
import torch.nn.functional as F
import transformers  # 需要安装 transformers 库以自动加载模型

# =================================================================
# 1. 核心功能模块
# =================================================================

def register_debug_hooks(model, prefix=""):
    """
    自动识别模型结构（HuggingFace 或自定义），并为 Embedding、每层 Block 和 LM Head 
    注册 Forward Hook，缓存输出用于后续比对。
    """
    cache = {}

    def _make_hook(name):
        def hook(module, input, output):
            # 兼容输出为 tensor 或 tuple/list 的情况
            out = output[0] if isinstance(output, (tuple, list)) else output
            cache[name] = out.detach()
        return hook

    # 1. 识别并挂接 Embedding 层
    if hasattr(model, "embed_tokens"):
        model.embed_tokens.register_forward_hook(_make_hook(f"{prefix}embed"))
    elif hasattr(model, "wte"):
        model.wte.register_forward_hook(_make_hook(f"{prefix}embed"))

    # 2. 识别并挂接 Transformer 每一层
    layers = getattr(model, "layers", None) or getattr(model, "h", None)
    if layers is not None:
        for i, layer in enumerate(layers):
            layer.register_forward_hook(_make_hook(f"{prefix}layer_{i}"))

    # 3. 挂接最终的 LM Head
    if hasattr(model, "lm_head"):
        model.lm_head.register_forward_hook(_make_hook(f"{prefix}lm_head"))

    return cache


def compare_tensors(a, b, name, atol=1e-5, rtol=1e-5):
    """
    对比两个张量的数值差异，返回详细的统计数据。
    """
    if a.shape != b.shape:
        print(f"  [❌ SHAPE MISMATCH] {name}: {a.shape} vs {b.shape}")
        return False

    diff = (a.float() - b.float()).abs()
    max_diff = diff.max().item()
    mean_diff = diff.mean().item()
    allclose = torch.allclose(a, b, atol=atol, rtol=rtol)

    # 输出单层的比对结果
    status = "✅" if allclose else "❌"
    print(f"  [{status}] {name:30s} | Max: {max_diff:.3e}, Mean: {mean_diff:.3e}")
    return allclose


def run_layerwise_diff_check(train_model, rollout_model, input_ids, atol=1e-5, rtol=1e-5):
    """
    主执行函数：固定同一输入，跑两边前向，逐层比对并定位首个分叉点。
    """
    # 统一进入 eval 模式并禁用自动微分
    train_model.eval()
    rollout_model.eval()

    train_cache = register_debug_hooks(train_model, prefix="train_")
    rollout_cache = register_debug_hooks(rollout_model, prefix="rollout_")

    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=False):
        _ = train_model(input_ids=input_ids)
        _ = rollout_model(input_ids=input_ids)

    # 提取所有需要比对的 Key，并按顺序排序
    keys_to_check = sorted(train_cache.keys())
    
    print("\n🔍 [开始逐层比对] ...")
    for k in keys_to_check:
        if k in rollout_cache:
            layer_name = k.replace("train_", "")
            match = compare_tensors(train_cache[k], rollout_cache[k], layer_name, atol, rtol)
            
            # 发现首个分叉点，立即终止
            if not match:
                print(f"\n🚨 [定位到首个分叉点]: {layer_name}")
                return False
        else:
            print(f"  ⚠️ [MISSING KEY] {k} 在 rollout 中未找到对应的输出")
            
    print("\n✅ [比对完成] 所有中间层数值完全一致（All Match）。")
    return True

# =================================================================
# 2. 命令行接口 (CLI)
# =================================================================

def main():
    parser = argparse.ArgumentParser(
        description="RL 训推 Hidden State 逐层 Diff 自动化排查脚本",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
使用示例:
  python debug_layerwise_diff.py \\
      --model_path /path/to/your_model \\
      --input_text "Hello, how are you" \\
      --atol 1e-5 \\
      --rtol 1e-5
        """
    )
    parser.add_argument("--model_path", type=str, required=True, help="模型路径")
    parser.add_argument("--input_text", type=str, required=True, help="用于排查的固定 Prompt")
    parser.add_argument("--atol", type=float, default=1e-5, help="绝对误差容忍度 (Absolute Tolerance)")
    parser.add_argument("--rtol", type=float, default=1e-5, help="相对误差容忍度 (Relative Tolerance)")
    
    args = parser.parse_args()

    print(f"🚀 加载模型: {args.model_path}")
    # 假设 rollout_model 是加载后未做任何优化的原始版本，或加载的本地推理权重
    # 在实际工程中，这里通常分别加载 "Training Engine Model" 和 "Rollout Engine Model"
    # 这里为了演示脚本完整性，使用同一模型，实际使用请替换为各自的模型实例
    model = transformers.AutoModelForCausalLM.from_pretrained(args.model_path, device_map="auto")
    
    # 如果你有两套模型（例如一套是训练态的，一套是推理态的），在这里分别加载
    # train_model = ...
    # rollout_model = ...

    print(f"📝 序列化输入文本: \"{args.input_text}\"")
    tokenizer = transformers.AutoTokenizer.from_pretrained(args.model_path)
    # 强制不添加特殊 token (如 <bos>, <eos>)，确保两边输入完全一致
    input_ids = tokenizer.encode(args.input_text, add_special_tokens=False, return_tensors="pt")
    
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
    
    # 执行自动化排查
    run_layerwise_diff_check(
        train_model=model, 
        rollout_model=model, # 实际使用时传入你的 rollout 模型
        input_ids=input_ids, 
        atol=args.atol, 
        rtol=args.rtol
    )

if __name__ == "__main__":
    main()

📥 使用方法
将模型文件和代码准备好后，直接在终端中运行即可。
1. 准备环境
确保服务器安装了 torch, transformers 等依赖。
2. 命令行运行

In [ ]:
python debug_layerwise_diff.py \
    --model_path /data/models/Qwen3-8B \
    --input_text "The quick brown fox jumps over" \
    --atol 1e-5 \
    --rtol 1e-5

3. 查看输出结果
脚本会打印出每一层的比对结果。
情形 A：发现分叉

In [ ]:
🔍 [开始逐层比对] ...
  [✅] embed                           | Max: 0.000e+00, Mean: 0.000e+00
  [✅] layer_0                         | Max: 1.073e-07, Mean: 1.254e-08
  [✅] layer_1                         | Max: 1.907e-06, Mean: 2.146e-07
  [❌] layer_2                         | Max: 3.815e-02, Mean: 1.243e-03

🚨 [定位到首个分叉点]: layer_2

情形 B：完美对齐

In [ ]:
🔍 [开始逐层比对] ...
  [✅] embed                           | Max: 0.000e+00, Mean: 0.000e+00
  [✅] layer_0                         | Max: 1.073e-07, Mean: 1.254e-08
  ...
  [✅] layer_28                        | Max: 9.537e-07, Mean: 8.447e-08
  [✅] lm_head                         | Max: 5.960e-06, Mean: 7.331e-07

✅ [比对完成] 所有中间层数值完全一致（All Match）。

💡 实际工程中的微调说明
在实际的 RLHF/RLAIF 工程中，Train Engine（如 DeepSpeed 或 Megatron）和 Rollout Engine（如 vLLM 或 SGLang）通常无法在同一个 .py 脚本里直接通过 hook 交互。

最佳实践是：
1. 训练侧：保留这套 hook 脚本。在训练暂停时，读取 Rollout 阶段生成的完整 token 序列（input_ids），在训练引擎中跑一次纯前向（no_grad()），把缓存到 GPU 显存里的 cache 用 torch.save 导出为 .pt 文件。
2. 推理侧：在 Rollout 引擎源码（或离线重跑前向时）加入同样的 hook，跑完生成同样名字的 .pt 缓存文件。
3. 本脚本作用：编写一个简单的 load_and_compare.py，分别加载这两个 .pt 文件，利用上面代码里的 compare_tensors 逻辑进行最终比对，即可在完全不影响原训练/推理框架性能的前提下，精准定位分叉点。

###  检查权重与并行配置
   - 确认训练权重和 rollout 权重完全同步，无 stale weight。
   - 检查 TP、DP、EP、PP 配置是否一致。
   - MoE 模型要确认 expert 数量、路由策略、top-k、负载平衡配置一致。


在 RL 训推不一致的排查流程中，“检查权重与并行配置” **位于“定位分叉点”之前**，属于最基础的“查户口”环节。

它的核心逻辑是：<font color='red'>“先确保物理条件相同，再谈计算逻辑”</font>。
  - **如果训练和推理连最底层的权重或并行切分方式都不一致，那么上层所有算子的对齐都毫无意义**。

以下为您详细拆解这一环节的三个核心检查项及其技术原理：
####  确认权重同步：消除“ stale weight”（过期权重）
在 RLHF/RLAIF 的迭代过程中，训练引擎（Training Engine）和推理引擎（Rollout Engine）是交替工作的。最常见的“低级错误”就是权重同步脱节。

##### 什么是 Stale Weight（过期权重）：
  - 在第 k 次迭代中，推理引擎使用的是第 k 版的权重 θk生成了 Rollout 序列。
  - 随后训练引擎完成了梯度更新，权重变成了 θk+1。
  - 如果在第 k+1 次迭代开始前，没有及时将 θk+1同步给推理引擎，推理引擎依然拿着 θk去做自回归生成，那么训练侧算出的 logprob 自然无法与推理侧对齐。
  
##### 排查要点：
  - 建立严格的权重同步机制。在每一次 RL 迭代开始时，必须强制从训练引擎将最新权重下发（下发方式可以是全量 Copy，也可以是增量 Refit）。
  - 检查权重版本号或 Hash 值，确保两端完全一致。
  
####  检查并行切分配置（TP / DP / EP / PP）
大模型通常依赖多卡甚至多机并行。不同的并行策略会彻底改变矩阵乘法（GEMM）的切分方式和张量形状。
  - TP（张量并行 Tensor Parallelism）：
    - 这是训推对齐中的头号杀手。如果训练时 TP=2，而推理时 TP=1，那么 MLP 层的 FFN 矩阵或者 Attention 的 QKV 矩阵就会被切成不同的子矩阵。
    - 由于 FP16/BF16 精度有限，大矩阵乘法的结果和几个小矩阵乘法结果再拼接的结果，在浮点数精度上绝对不是 100% 相等的。因此，训推必须使用相同的 TP 策略，甚至相同的 TP 切分维度（如 Row/Col split）。
  - DP（数据并行 Data Parallelism）：
    - 主要影响 Batch 的切分。如果训练侧因为 DP 切分导致某个 Batch 需要 Padding 到特定长度，而推理侧没有，Attention Mask 的计算就会出现偏差。
  - EP（专家并行 Expert Parallelism）与 PP（流水线并行 Pipeline Parallelism）：
    - 同样会影响张量的物理布局。必须确保训练和推理引擎的 world_size、rank 以及对应的切分配置完全映射一致。
    
####  MoE 模型专属：对齐专家路由与负载均衡
对于 MoE（Mixture of Experts）模型，权重同步和并行配置变得更加复杂。即使总权重完全一致，只要“派活”的机制不对，训练和推理就会走上完全不同的计算路径。
  - Expert 数量与物理布局：
    - 检查推理侧加载的 Expert 总数、每个 Node/GPU 上分配的 Expert 数量是否与训练侧完全一致。
  - 路由策略（Routing Strategy）：
    - MoE 的路由器（Router/Expert Gate）本质上是一个 Softmax + Top-K 分类器。它决定了某个 token 该发送给哪几个 Expert。
    - 如果训练和推理的 Router 精度不同，或者随机种子不同，导致同一个 token 在第 i 层被分配到了 Expert A（推理侧），而在第 i 层被分配到了 Expert B（训练侧），那么整个后续计算就会彻底分叉。
  - Top-k 与负载均衡（Load Balance）：
    - Top-k：必须确保选取的专家数量 k 一致。
    - 负载均衡：为了防止 Expert 负载不均，路由算法中通常带有负载平衡 Loss（如 Switch Transformer 或 GShard 的 load balancing loss）。如果训练侧和推理侧在路由阶段开启了不同的负载均衡阈值或随机噪声（如 Gumbel Noise），就会导致选出的 Expert 组合发生微妙的变化。
    
#### 总结
“检查权重与并行配置”就像是在赛车前检查两个车手的车辆：
  - 车辆引擎是否完全一样？（权重是否同步）
  - 底盘调教和轮胎切分策略是否一致？（TP/DP/EP 并行配置）
  - 自动驾驶的换胎逻辑是否对齐？（MoE 路由与 Top-k）

###  检查 Attention 路径
   - 训练 prefill 与推理 prefill/decode 是否使用同一套 attention 语义。
   - mask 实现是否一致：加极小值 mask 和 set-mask 不参与计算会产生差异。
   - FA、PFA、PagedAttention 的 reduce 轴切分顺序是否一致。
   - 在线 softmax 分块大小、归约节奏是否一致。
###  检查精度路径
   - 训练和推理的输入精度、激活精度、logits 精度是否一致。
   - BF16 尾数位较少，容易在自回归累积中放大舍入误差；FP16 尾数更细，但动态范围更小，需配合 loss scaling。
   - LM head softmax、KV cache、router logits 等敏感位置建议单独核对精度。
###  检查 KV Cache 与 RoPE
   - KV cache 初始化精度、更新方式、concat/append 顺序是否一致。
   - RoPE 实现是否与推理引擎完全一致，尤其是不同 backend 下常有细微差异。
   - 长序列下 decode 阶段误差更容易累积，需要重点验证。
###  检查 Normalization 与融合算子
   - RMSNorm、LayerNorm、SiLU MLP 等融合算子在训练和推理中是否等价。
   - 推理端常用 fused kernel，训练端可能是分步算子；中间截断顺序不同就会产生 diff。
   - 必要时替换为同一套 batch-invariant kernel，或为推理前向补写训练反向。
###  检查 MoE 路由
   - router 计算精度是否足够高。
   - top-k 选择是否稳定；torch.topk 等非稳定排序可能导致不同 run 选不同 expert。
   - token permutation、expert 累加顺序是否确定。
   - 可采用 rollout routing replay，训练时复用推理阶段记录的路由路径。
###  检查采样与 logprob 计算
   - rollout 使用的 temperature、top-p、top-k、mask <font color='red'>是否与训练重算时一致。</font>
   - logprob 是否在同一引擎、同一精度、同一 vocab mask 下计算。
   - 如果训推未对齐，直接用 rollout probs 做优化会有偏；<font color='red'>很多框架因此退回到用训练引擎重算 training probs。</font>
###  检查随机性与非确定性
   - 关闭或固定 dropout、dropout mask、随机 seed。
   - 确认 CUDA/算子 deterministic 模式是否开启。
   - 检查数据并行采样、序列打包、padding mask 是否一致。
###  算法层兜底
<font color='red'>如果工程对齐暂时做不到完全归零，可叠加算法侧缓解</font>：重要性采样截断/掩码、得分中心化、路由回放、精度切换等。 但这些更适合缓解残余 diff，不能替代底层数值路径对齐。

## 实操建议
推荐按这个顺序排查：先逐层定位 diff 起点 → 再核对精度和算子实现 → 再看并行与路由 → 最后检查采样与 logprob 计算。

如果目标是高质量 RL 训练，最好把 logdiff 稳定压到接近 0，并直接复用 rollout probs；这样既能减少梯度偏差，也能避免训练阶段重复前向带来的额外开销。

### 下面是一段可直接套用的 PyTorch 调试模板，
- 核心思路是：
  - <font color='red'>在训练模型和推理模型上注册逐层 hook</font>，固定同一输入，对比 embedding、每层 hidden state、attention logits 和最终 logits。

In [ ]:
import torch
import torch.nn.functional as F

def register_debug_hooks(model, prefix=""):
    """给模型注册逐层 hook，自动收集 hidden state 与 attention logits。"""
    cache = {}

    def _make_hook(name):
        def hook(module, input, output):
            # output 可能是 tuple，取第一个 tensor
            out = output[0] if isinstance(output, (tuple, list)) else output
            cache[name] = out.detach()
        return hook

    # 1. Embedding 层
    if hasattr(model, "embed_tokens"):
        model.embed_tokens.register_forward_hook(_make_hook(f"{prefix}embed"))
    elif hasattr(model, "wte"):
        model.wte.register_forward_hook(_make_hook(f"{prefix}embed"))

    # 2. Transformer 层
    layers = getattr(model, "layers", None) or getattr(model, "h", None)
    if layers is not None:
        for i, layer in enumerate(layers):
            layer.register_forward_hook(_make_hook(f"{prefix}layer_{i}"))

    # 3. LM Head / logits
    if hasattr(model, "lm_head"):
        model.lm_head.register_forward_hook(_make_hook(f"{prefix}lm_head"))

    return cache


def compare_tensors(a, b, name="tensor", atol=1e-4, rtol=1e-5):
    """对比两个 tensor，返回 diff 统计。"""
    if a is None and b is None:
        return None

    if a is None or b is None:
        print(f"[MISSING] {name}: 一方缺失")
        return None

    if a.shape != b.shape:
        print(f"[SHAPE MISMATCH] {name}: {a.shape} vs {b.shape}")
        return None

    diff = (a - b).float()
    abs_diff = diff.abs()
    max_diff = abs_diff.max().item()
    mean_diff = abs_diff.mean().item()
    rms_diff = (diff.pow(2).mean().sqrt()).item()
    eq = torch.allclose(a, b, atol=atol, rtol=rtol)

    print(
        f"[DIFF] {name:30s} | max={max_diff:.3e} mean={mean_diff:.3e} "
        f"rms={rms_diff:.3e} allclose(atol={atol}, rtol={rtol})={eq}"
    )
    return {"name": name, "max": max_diff, "mean": mean_diff, "rms": rms_diff, "allclose": eq}


def compare_logprobs(logits_train, logits_rollout, labels=None, vocab_mask=None):
    """对比训练侧与 rollout 侧的 logprob。"""
    if vocab_mask is not None:
        logits_train = logits_train.masked_fill(~vocab_mask, -1e9)
        logits_rollout = logits_rollout.masked_fill(~vocab_mask, -1e9)

    p_train = F.log_softmax(logits_train.float(), dim=-1)
    p_rollout = F.log_softmax(logits_rollout.float(), dim=-1)

    if labels is not None:
        lp_train = torch.gather(p_train, -1, labels.unsqueeze(-1)).squeeze(-1)
        lp_rollout = torch.gather(p_rollout, -1, labels.unsqueeze(-1)).squeeze(-1)
    else:
        lp_train = p_train
        lp_rollout = p_rollout

    diff = (lp_train - lp_rollout).abs()
    print(
        f"[LOGPROB DIFF] max={diff.max().item():.3e} "
        f"mean={diff.mean().item():.3e} "
        f"p95={diff.flatten().quantile(0.95).item():.3e}"
    )
    return diff


def run_layerwise_diff_check(
    train_model,
    rollout_model,
    input_ids,
    attention_mask=None,
    position_ids=None,
    atol=1e-4,
    rtol=1e-5,
    stop_at_first_mismatch=False,
):
    """
    逐层对比训练模型与 rollout 模型的 hidden state。
    使用前请确保两个模型处于 eval 模式，且权重一致。
    """
    train_model.eval()
    rollout_model.eval()

    train_cache = register_debug_hooks(train_model, prefix="train_")
    rollout_cache = register_debug_hooks(rollout_model, prefix="rollout_")

    with torch.no_grad(), torch.autocast("cuda", enabled=False):
        kwargs = {"input_ids": input_ids}
        if attention_mask is not None:
            kwargs["attention_mask"] = attention_mask
        if position_ids is not None:
            kwargs["position_ids"] = position_ids

        # 如果模型支持 output_hidden_states / output_attentions，也可以同时打开
        _ = train_model(**kwargs)
        _ = rollout_model(**kwargs)

    # 逐层对比
    train_keys = sorted(k for k in train_cache if k.startswith("train_layer_") or k == "train_embed")
    rollout_keys = sorted(k for k in rollout_cache if k.startswith("rollout_layer_") or k == "rollout_embed")

    pairs = []
    for tk, rk in zip(train_keys, rollout_keys):
        layer_name = tk.replace("train_", "")
        pairs.append((tk, rk, layer_name))

    results = []
    for tk, rk, layer_name in pairs:
        r = compare_tensors(
            train_cache[tk],
            rollout_cache[rk],
            name=layer_name,
            atol=atol,
            rtol=rtol,
        )
        if r is not None:
            results.append(r)
            if stop_at_first_mismatch and not r["allclose"]:
                print(f"[STOP] 首次分叉点: {layer_name}")
                break

    # 最终 logits 对比
    if "train_lm_head" in train_cache and "rollout_lm_head" in rollout_cache:
        compare_tensors(
            train_cache["train_lm_head"],
            rollout_cache["rollout_lm_head"],
            name="lm_head_logits",
            atol=atol,
            rtol=rtol,
        )

    return results